# Limpieza de datos

Se realiza la limpieza de datos y se genera la tabla de ventas en schema de silver

In [0]:
T_BRONZE = 'blur_proyecto_final.bronze.sales_raw'
T_SILVER = 'blur_proyecto_final.silver.sales_silver'


In [0]:
T_BRONZE = 'blur_proyecto_final.bronze.sales_raw'
df_sales = spark.read.table(T_BRONZE)
filas_bronze = df_sales.count()
print(filas_bronze)

In [0]:
display(spark.table(f"{T_BRONZE}").limit(5))

## Se consulta el ESQUEMA
Se verifica el esquema para revisar los campos y tipos de datos

In [0]:
df_sales.printSchema()

## # Contar valores NULL
Contar cuantos valores null hay por cada columna

In [0]:
from pyspark.sql import functions as F

df_sales.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_sales.columns
]).show()

In [0]:
from pyspark.sql import functions as F
df_silver_ventas = ( df_sales                    
                     .select(
                        F.col("factura").alias("factura"),
                        F.col("cliente").alias("cliente"),
                        F.col("cantidad").alias("cantidad"),
                        F.col("fecha").alias("fecha"),
                        F.col("monto").alias("monto")
                        )
                      .withColumn("factura",F.regexp_replace(F.col("factura"), "^C", "") ) 
                      .withColumn("monto", F.regexp_replace(F.col("monto"), "[,]", "."))
                      .withColumn("monto", F.col("monto").cast("double") )                    
                      .withColumn("fecha", F.to_timestamp(F.col("fecha"), "M/d/yyyy H:mm:ss"))
                      .withColumn("date", F.to_date(F.col("fecha")) )
                      .withColumn("cliente", F.coalesce(F.col("cliente"), F.lit("sin_cliente")))
                      .withColumn("cantidad",F.col("cantidad").cast("int") )
                      .withColumn("fecha_ingesta", F.current_timestamp())
                      .withColumn("usuario_ingesta", F.current_user())
                      .withColumn("origen",F.lit("kaggle"))
)

 
df_silver_ventas.show(5)
df_silver_ventas.printSchema()






## Creacion tabla en Silver
Despues de la limpieza de los datos se crea la tabla en silver

In [0]:
df_silver_ventas.write.mode("overwrite").saveAsTable(f"{T_SILVER}")
